# Preparación del Entorno

In [4]:
import pandas as pd
df = pd.read_csv("S07_PD_Diaz_DatasetSinOutliers.csv")

In [5]:
df.shape

(589386, 26)

Como los valores de df.shape son los mismos de la semana pasada, podemos continuar

In [6]:
df[["edad_cliente", "precio_unitario", "monto_compra", "unidades_vendidas"]].describe()

,edad_cliente,precio_unitario,monto_compra,unidades_vendidas
count,589386.000000,589386.000000,5.893860e+05,589386.000000
mean,46.533572,344097.306010,6.542166e+05,1.878750
std,16.231314,196498.181671,6.300968e+05,1.147132
min,18.000000,15100.000000,0.000000e+00,1.000000
25%,33.000000,181200.000000,2.214000e+05,1.000000
50%,47.000000,311600.000000,4.554000e+05,1.000000
75%,60.000000,481500.000000,8.516000e+05,2.000000
max,75.000000,899500.000000,4.450000e+06,5.000000


## Rangos antes de escalar
Se puede evidenciar que la columna monto_compra tiene el rango más alto, mientras que edad_cliente tiene el rango más bajo

# Preparación para un futuro modelo

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
df_entrenamiento, df_prueba = train_test_split(df, test_size=0.2, random_state=42)

In [9]:
suma_df = len(df_entrenamiento) + len(df_prueba)
print(suma_df)

589386


Se puede ver que la suma de los datos en entrenamiento y prueba son los mismos que en el dataset de la semana pasada.

## Por qué evito la fuga de datos
Para evitar la fuga de datos (data leakage), el escalador se ajusta únicamente utilizando df_entrenamiento. Esto es importante porque, si se calcula el escalador utilizando conjuntamente los datos de entrenamiento y de prueba, se estaría incorporando información del conjunto de prueba durante la etapa de entrenamiento.

Como consecuencia, la evaluación del modelo sobre los datos de prueba dejaría de ser completamente confiable, ya que estos datos habrían influido indirectamente en el proceso de transformación. Esto genera una contaminación del conjunto de prueba y puede producir métricas de rendimiento artificialmente optimistas.

Por esta razón, el escalador se ajusta (fit) exclusivamente con los datos de entrenamiento y posteriormente se aplica (transform) tanto al conjunto de entrenamiento como al conjunto de prueba, utilizando exactamente los mismos parámetros aprendidos durante el entrenamiento.

In [10]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

In [16]:
escalador_edad = StandardScaler()
escalador_monto = RobustScaler()
escalador_precio = StandardScaler()
escalador_unidades = StandardScaler()

escalador_edad.fit(df_entrenamiento[["edad_cliente"]])
escalador_monto.fit(df_entrenamiento[["monto_compra"]])
escalador_precio.fit(df_entrenamiento[["precio_unitario"]])
escalador_unidades.fit(df_entrenamiento[["unidades_vendidas"]])

StandardScaler()

In [17]:
escalador_edad.mean_

array([46.52698364])

In [18]:
df_entrenamiento["edad_cliente_escalada"] = escalador_edad.transform(df_entrenamiento[["edad_cliente"]])
df_prueba["edad_cliente_escalada"] = escalador_edad.transform(df_prueba[["edad_cliente"]])

df_entrenamiento["monto_compra_escalada"] = escalador_monto.transform(df_entrenamiento[["monto_compra"]])
df_prueba["monto_compra_escalada"] = escalador_monto.transform(df_prueba[["monto_compra"]])

df_entrenamiento["precio_unitario_escalada"] = escalador_precio.transform(df_entrenamiento[["precio_unitario"]])
df_prueba["precio_unitario_escalada"] = escalador_precio.transform(df_prueba[["precio_unitario"]])

df_entrenamiento["unidades_vendidas_escalada"] = escalador_unidades.transform(df_entrenamiento[["unidades_vendidas"]])
df_prueba["unidades_vendidas_escalada"] = escalador_unidades.transform(df_prueba[["unidades_vendidas"]])

In [19]:
df_entrenamiento.filter(like="_escalada").describe()
df_prueba.filter(like="_escalada").describe()

,edad_cliente_escalada,monto_compra_escalada,precio_unitario_escalada,unidades_vendidas_escalada
count,117878.000000,117878.000000,117878.000000,117878.000000
mean,0.002030,0.316536,0.001385,-0.001860
std,1.001485,0.999130,0.998460,0.997664
min,-1.758051,-0.722725,-1.673510,-0.766056
25%,-0.833636,-0.370653,-0.827452,-0.766056
50%,0.029151,0.002064,-0.165054,-0.766056
75%,0.830310,0.633754,0.701863,0.105277
max,1.754725,6.242496,2.822858,2.719276


Los valores son coherentes para cada uno de los escaladores